# Parte 3 - Cruce por ubigeo y análisis
**Pregunta:** ¿El Estado declara emergencia donde más llueve? (Temporada 2024)

In [1]:
import pandas as pd

### Paso 1 - Crear datos/ubigeos.csv
Usamos los ubigeos departamentales oficiales del INEI.
Escribimos el ubigeo como **texto** ("01", no 1) para no perder el cero.

In [5]:
#Ubigeos oficiales del INEI.
ubigeos = pd.DataFrame({"ubigeo":["01","02","03","04","05","06","07","08","09","10","11","12","13","14","15","16","17","18","19","20","21","22","23","24","25"], "departamento": ["Amazonas","Áncash","Apurímac","Arequipa","Ayacucho","Cajamarca","Callao","Cusco","Huancavelica","Huánuco","Ica","Junín","La Libertad","Lambayeque","Lima","Loreto","Madre de Dios","Moquegua","Pasco","Piura","Puno","San Martín","Tacna","Tumbes","Ucayali"]})
#index=False: no guardar el índice; utf-8-sig: tildes correctas en excel
ubigeos.to_csv("datos/ubigeos.csv", index=False, encoding="utf-8-sig")
print("filas:", len(ubigeos)) #control: deben ser 25 departamentos
ubigeos

filas: 25


,ubigeo,departamento
0,01,Amazonas
1,02,Áncash
2,03,Apurímac
3,04,Arequipa
4,05,Ayacucho
5,06,Cajamarca
6,07,Callao
7,08,Cusco
8,09,Huancavelica
9,10,Huánuco


### Paso 2 - Leer "ubigeos.csv" dos veces
primero con pd.read_csv normal y luego indicando "dtype={"ubigeo":str}

In [15]:
ubigeos_normal=pd.read_csv("datos/ubigeos.csv")
print(ubigeos_normal.dtypes)
ubigeos_normal.head(10)

ubigeo          int64
departamento      str
dtype: object


,ubigeo,departamento
0,1,Amazonas
1,2,Áncash
2,3,Apurímac
3,4,Arequipa
4,5,Ayacucho
5,6,Cajamarca
6,7,Callao
7,8,Cusco
8,9,Huancavelica
9,10,Huánuco


In [16]:
ubigeos_texto=pd.read_csv("datos/ubigeos.csv", dtype={"ubigeo": str})
print(ubigeos_texto.dtypes)
ubigeos_texto.head(10)

ubigeo          str
departamento    str
dtype: object


,ubigeo,departamento
0,01,Amazonas
1,02,Áncash
2,03,Apurímac
3,04,Arequipa
4,05,Ayacucho
5,06,Cajamarca
6,07,Callao
7,08,Cusco
8,09,Huancavelica
9,10,Huánuco


In [17]:
#comparamos ambas lecturas en una sola tabla
comparacion=pd.DataFrame({"departamento": ubigeos_texto["departamento"],"ubigeo_normal":ubigeos_normal["ubigeo"],"ubigeo_texto":ubigeos_texto["ubigeo"]})
print(comparacion.head(10))
#prueba concreta: buscar Amazonas por su ubigeo "01"
print("\n¿Encuentra '01' leyendo normal?:", (ubigeos_normal["ubigeo"] == "01").sum())
print("¿Encuentra '01' leyendo como texto?:", (ubigeos_texto["ubigeo"] == "01").sum())

   departamento  ubigeo_normal ubigeo_texto
0      Amazonas              1           01
1        Áncash              2           02
2      Apurímac              3           03
3      Arequipa              4           04
4      Ayacucho              5           05
5     Cajamarca              6           06
6        Callao              7           07
7         Cusco              8           08
8  Huancavelica              9           09
9       Huánuco             10           10

¿Encuentra '01' leyendo normal?: 0
¿Encuentra '01' leyendo como texto?: 1


##### ¿Por qué es importante leer el ubigeo como texto?
- **Qué pasó:** al leer el archivo de forma normal, pandas convirtió el ubigeo de Amazonas "01" en 1. Los departamentos del 01 al 09 perdieron el cero de la izquierda.
- **Qué es el ubigeo:** es un código de identificación, debe guardarse como texto.
- **La prueba:** al buscar "01", la lectura normal encontró 0 coincidencias, mientras que la lectura como texto encontró 1 (Amazonas). Si cruzamos tablas con el ubigeo como número, Amazonas se perdería.
- **Formato oficial:** el INEI usa siempre dos dígitos "01", así debe mantenerse en la tabla_final.csv
- **Decisión:** leer siempre con: `dtype={"ubigeo": str}`

### Paso 3 - Agregar el ubigeo a cada tabla
Agregamos el `ubigeo` buscando el nombre del departamento en `ubigeos.csv`.
Usamos `how=left` para no perder ningún departamento: si alguno no encuentra su ubigeo, queda vacío (NaN) y lo detectamos en el paso 4.

### 3a. Decretos por departamento

In [ ]:
#Leemos los ubigeos SIEMPRE como texto (decisión del paso 2)
ubigeos = pd.read_csv("datos/ubigeos.csv", dtype={"ubigeo": str})
#Tabla de la Parte 1: declaratorias y prórrogas por departamento
decretos=pd.read_csv("datos/decretos_por_departamento.csv")
print("Decretos:", decretos.shape) #control: 25 filas, 3 columnas
#Agregamos el ubigeo buscando por nombre: how="left" para que no se pierdan los departamentos que no tengan decretos
decretos_ub = decretos.merge(ubigeos, how="left", on="departamento")
#Ordenamos las columnas para que ubigeo quede primero
decretos_ub = decretos_ub[["ubigeo", "departamento", "declaratorias", "prorrogas"]]
decretos_ub.head()

Decretos: (25, 3)


,ubigeo,departamento,declaratorias,prorrogas
0,01,Amazonas,4,2
1,02,Áncash,5,5
2,03,Apurímac,7,4
3,04,Arequipa,7,6
4,05,Ayacucho,7,5


### 3b. Lluvias por departamento

In [ ]:
# Tabla de la Parte 2: lluvia de la temporada en la capital de cada departamento
lluvias = pd.read_csv("datos/lluvias_por_departamento.csv")
print("Lluvias:", lluvias.shape) # control: 25 filas, 6 columnas
# Agregamos el ubigeo buscando por nombre, igual que con decretos
lluvias_ub = lluvias.merge(ubigeos, how="left", on="departamento")
lluvias_ub = lluvias_ub[["ubigeo", "departamento", "capital", "latitud", "longitud",
                         "lluvia_total_mm", "dias_lluvia_fuerte"]]
lluvias_ub.head()

Lluvias: (25, 6)


,ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,01,Amazonas,Chachapoyas,-6.23169,-77.86903,455.0,0
1,02,Áncash,Huaraz,-9.52614,-77.52869,1058.3,3
2,03,Apurímac,Abancay,-13.63426,-72.88380,670.6,2
3,04,Arequipa,Arequipa,-16.39899,-71.53747,235.3,1
4,05,Ayacucho,Ayacucho,-13.16380,-74.22345,467.5,1


### Paso 4 - Verificación: ¿algún departamento no encontró su ubigeo?
### 4a. Decretos

In [ ]:
# Si un nombre está escrito distinto (sin tilde, por ejemplo), su ubigeo queda vacío (NaN)
sin_ubigeo = decretos_ub[decretos_ub["ubigeo"].isna()]
print("Departamentos de decretos sin ubigeo:", len(sin_ubigeo))
print(sin_ubigeo)
# Revisión al revés: ¿algún ubigeo oficial no aparece en decretos?
faltan = set(ubigeos["departamento"]) - set(decretos["departamento"])
print("Departamentos oficiales que no están en decretos:", faltan)
# Control final
print("Filas:", len(decretos_ub), "| ubigeos vacíos:", decretos_ub["ubigeo"].isna().sum())

Departamentos de decretos sin ubigeo: 0
Empty DataFrame
Columns: [ubigeo, departamento, declaratorias, prorrogas]
Index: []
Departamentos oficiales que no están en decretos: set()
Filas: 25 | ubigeos vacíos: 0


**Resultado 4a:** los 25 departamentos de `decretos_por_departamento.csv` encontraron su ubigeo.

### 4b. Verificación de lluvias

In [27]:
sin_ubigeo_ll = lluvias_ub[lluvias_ub["ubigeo"].isna()]
print("Departamentos de lluvias sin ubigeo:", len(sin_ubigeo_ll))
print(sin_ubigeo_ll)
print("Oficiales que no están en lluvias:", set(ubigeos["departamento"]) - set(lluvias["departamento"]))
print("Filas:", len(lluvias_ub), "| ubigeos vacíos:", lluvias_ub["ubigeo"].isna().sum())

Departamentos de lluvias sin ubigeo: 0
Empty DataFrame
Columns: [ubigeo, departamento, capital, latitud, longitud, lluvia_total_mm, dias_lluvia_fuerte]
Index: []
Oficiales que no están en lluvias: set()
Filas: 25 | ubigeos vacíos: 0


Resultado 4b: los 25 departamentos de la tabla de lluvias encontraron su ubigeo. No se corrigió nombres ya que en la Parte 2 se usa la misma lista oficial con tildes. El Callao, fue agregado en la Parte 2 y encontró su ubigeo (07).

### Paso 5 - Cruce por ubigeo
Partimos de la lista oficial de 25 ubigeos y le pegamos las dos tablas con `how="left"`.
Cruzamos por `ubigeo` (no por nombre) y quitamos la columna `departamento` de las otras tablas para que no se repita.